In [58]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from anngeno import AnnGeno
# pl.Config.set_tbl_rows(15)

In [59]:
# Configuration and paths
or_threshold_pheno = 0.95
or_threshold_anno = 0.95

eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

maf=1e-3
n = !wc -l $eur_samples_path
n_eur = int(n[0].split(' ')[0])
mac = maf*(2*n_eur)

config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Read the variant consequence configuration
records = []
for group, consequences in config["variant_consequences"].items():
    for consequence in consequences:
        records.append({'variant_class': group, 'vep_consequence': consequence})
var_cons = pl.DataFrame(records)


# Use a list comprehension to flatten the nested dictionary into records
records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records)
all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i64
"""plof""","""loftee_hc""","""#DD4344""","""LOFTEE HC""",1
"""missense""","""am_pathogenicity""","""#feb72d""","""AlphaMissense""",1
"""missense""","""esmscoremissense""","""#feb72d""","""ESM1v""",-1
"""genetic_diversity""","""cadd_raw""","""#1f77b4""","""CADD Raw""",1
"""conservation""","""gpn_star_llr_calibrated_mean""","""#942c80""","""GPN-Star""",-1
…,…,…,…,…
"""splicing""","""absplice_dna_max""","""#28a745""","""AbSplice (max)""",1
"""splicing""","""absplice2_max""","""#28a745""","""AbSplice2 (max)""",1
"""regulatory_nondir""","""promoterai_abs""","""#00A99D""","""PromoterAI abs""",1


In [60]:
exp_annos_df = pl.DataFrame({
    "category": ['non-coding indel', 'non-coding indel', 'non-coding indel', 'non-coding indel'],
    'annotation': ['noncoding_indel', 'intronic_indel', 'utr5_indel', 'utr3_indel'],
    "color": ['gray', 'gray', 'gray', 'gray'],
    'label': ['non-coding indel', 'non-coding indel', 'non-coding indel', 'non-coding indel'],
    'annotation_dir': [1, 1, 1, 1],
})

anno_config_df = pl.concat([anno_config_df, exp_annos_df])

In [ ]:
anno = pl.scan_parquet("/home/dnanexus/data_dir/genebass394genes_olink371genes_variants_union_annotated_genocode_251103.parquet")

# existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

existing_annos = ['loftee_hc', 'am_pathogenicity', 'noncoding_indel', 'intronic_indel', 'utr5_indel', 'utr3_indel']

# Filter variant classes
variant_class = "coding"
consequences_for_group = var_cons.filter(
    pl.col('variant_class') == variant_class
)['vep_consequence'].to_list()
filter_expression = pl.any_horizontal(
    (pl.col(c) == 1) for c in consequences_for_group if c in anno.collect_schema().names()
)

min_range = -2000
max_range = +0

anno = (
    anno
    .with_columns(
        noncoding_indel = (pl.col('vep_gencode_non_coding')==True) & ((pl.col('ref').str.len_chars()!=1) | (pl.col('alt').str.len_chars()!=1))
    )
    .with_columns(
        intronic_indel = pl.col('noncoding_indel') & (pl.col('consequence_intron_variant')==1),
        utr5_indel = pl.col('noncoding_indel') & (pl.col('consequence_5_prime_utr_variant')==1),
        utr3_indel = pl.col('noncoding_indel') & (pl.col('consequence_3_prime_utr_variant')==1),
    )
    .select(
        set(['id', 'region', 'TSS', 'Strand', 'gene_length', 'gene_name', 'dist_to_tss', 'vep_gencode_non_coding']).union(set(existing_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

utr3_indel,gene_name,region,intronic_indel,am_pathogenicity,noncoding_indel,TSS,Strand,dist_to_tss,vep_gencode_non_coding,loftee_hc,gene_length,utr5_indel,id
bool,str,str,bool,f32,bool,i64,cat,i64,bool,i8,i64,bool,str
false,"""LRP2""","""ENSG00000081479""",false,0.0,false,169362534,"""-""",165367,true,0,235427,false,"""chr2:169197167:T:G"""
false,"""LRP2""","""ENSG00000081479""",false,0.0,false,169362534,"""-""",158242,true,0,235427,false,"""chr2:169204292:A:T"""
false,"""ABCB11""","""ENSG00000073734""",false,0.0,false,169031324,"""-""",793,true,0,115828,false,"""chr2:169030531:C:T"""
false,"""ABCB11""","""ENSG00000073734""",false,0.0,false,169031324,"""-""",120499,true,0,115828,false,"""chr2:168910825:T:G"""
false,"""ABCB11""","""ENSG00000073734""",false,0.0607,false,169031324,"""-""",61780,false,0,115828,false,"""chr2:168969544:T:C"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
false,"""ZNF770""","""ENSG00000198146""",false,0.0,true,34988287,"""-""",13678,true,0,9948,false,"""chr15:34974609:ACTTCT:A"""
false,"""SETX""","""ENSG00000107290""",false,0.0,false,132354986,"""-""",-3584,true,0,93632,false,"""chr9:132358570:C:A"""
false,"""DLC1""","""ENSG00000164741""",true,0.0,true,13604610,"""-""",403624,true,0,521251,false,"""chr8:13200986:C:CA"""


In [62]:
melted_anno = (
    anno
    .with_columns(
        am_pathogenicity = pl.col('am_pathogenicity') >= 0.95
    )

    .unpivot(
        index=["id", "region"],
        on=existing_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

    # Merge with annotation configuration to get direction and filter
    .join(
        anno_config_df,
        on="annotation",
        how="left"
    )
    .with_columns(
        annotation_score_dircor = pl.col('annotation_score') * pl.col("annotation_dir")
    )
)

melted_anno

id,region,annotation,annotation_score,category,color,label,annotation_dir,annotation_score_dircor
str,str,str,f32,str,str,str,i64,f64
"""chr2:169197167:T:G""","""ENSG00000081479""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr2:169204292:A:T""","""ENSG00000081479""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr2:169030531:C:T""","""ENSG00000073734""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr2:168910825:T:G""","""ENSG00000073734""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr2:168969544:T:C""","""ENSG00000073734""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
…,…,…,…,…,…,…,…,…
"""chr15:34974609:ACTTCT:A""","""ENSG00000198146""","""utr3_indel""",0.0,"""non-coding indel""","""gray""","""non-coding indel""",1,0.0
"""chr9:132358570:C:A""","""ENSG00000107290""","""utr3_indel""",0.0,"""non-coding indel""","""gray""","""non-coding indel""",1,0.0
"""chr8:13200986:C:CA""","""ENSG00000164741""","""utr3_indel""",0.0,"""non-coding indel""","""gray""","""non-coding indel""",1,0.0


In [63]:
appv = pl.scan_parquet("/home/dnanexus/data_dir/appv_files/avg_pheno_per_var_quantitative_EUR_genebass1e6_PRScorr_with_percentiles.parquet")

# Create a lazy frame with the unique keys
anno_keys = anno.select(pl.col('id').unique()).lazy()

# Chain the filter and the much faster semi join
appv = (
    appv
        .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        pl.col('n_individuals') <= mac
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value_ptile', 'n_individuals']
    )
)

# unique_phenotypes = appv.select('phenotype').unique().collect(engine='streaming').to_series()

In [64]:
# Get gene trait associations

plof = pl.read_parquet('/home/dnanexus/data_dir/association_files/rvat_EUR_500k_regenie.parquet').with_columns(
    phenotype = (pl.col('trait') + '_int'),
    region = pl.col('gene_id'),
    rvat_pval = (10** -pl.col("neg_log10p")),
).filter(
    (pl.col('trait_type') == 'quantitative')
)

loftee_corr = pl.read_parquet("/home/dnanexus/data_dir/association_files/loftee_correlation_quantitative_wgs_EUR_genebass1e6_maf1e3_snp_consistent.parquet")

gene_trait_df = loftee_corr.join(plof[['region', 'phenotype', 'beta', 'rvat_pval']], on=['region', 'phenotype'], how='inner').filter(
    pl.col('loftee_corr')*pl.col('beta') > 0
).with_columns(
    corr_dir = pl.col('loftee_corr')/pl.col('loftee_corr').abs()
)

# gene_trait_df = gene_trait_df.head()
gene_trait_df

region,gene_name,phenotype,loftee_corr,n_variants,beta,rvat_pval,corr_dir
str,str,str,f64,u64,f64,f64,f64
"""ENSG00000116183""","""PAPPA2""","""arm_fatfree_mass_right_int""",-0.014747,82603,-0.195409,9.1637e-8,-1.0
"""ENSG00000129083""","""COPB1""","""arm_fatfree_mass_right_int""",-0.014338,13594,-0.397279,0.006626,-1.0
"""ENSG00000100578""","""KIAA0586""","""arm_fatfree_mass_right_int""",-0.010787,28541,-0.058463,0.000305,-1.0
"""ENSG00000157766""","""ACAN""","""arm_fatfree_mass_right_int""",-0.036259,20244,-0.436618,6.0395e-11,-1.0
"""ENSG00000140443""","""IGF1R""","""arm_fatfree_mass_right_int""",-0.013644,82469,-0.412619,2.8609e-8,-1.0
…,…,…,…,…,…,…,…
"""ENSG00000112077""","""RHAG""","""reticulocyte_count_int""",0.033239,9849,0.952411,1.8038e-41,1.0
"""ENSG00000029534""","""ANK1""","""reticulocyte_count_int""",0.02685,54620,0.656849,2.1682e-10,1.0
"""ENSG00000197969""","""VPS13A""","""reticulocyte_count_int""",0.003971,55477,0.176348,2.1188e-7,1.0


In [65]:
gene_trait_df['region'].n_unique(), gene_trait_df.shape[0]

(352, 1176)

In [66]:
# --- 1. Lazily prepare the filter keys ---
region_keys = gene_trait_df.lazy().select(pl.col('region').unique())
anno_ids_lazy = melted_anno.lazy().join(
    region_keys, on='region', how='semi'
).select(pl.col('id').unique())


# --- Build main query (same as before, but stop before group_by) ---
gp_lazy = (
    appv
    .join(anno_ids_lazy, on="id", how="semi")
    .join(
        melted_anno.lazy().drop([c for c in melted_anno.columns if 'is_nan' in c]), 
        on="id", 
        how="inner"
    )
    .join(
        gene_trait_df.lazy(), 
        on=["region", "phenotype"], 
        how="inner"
    )
    .with_columns(
        mean_pheno_value_dircor_ptile = pl.when(pl.col('corr_dir') == -1)
            .then(1 - pl.col('mean_pheno_value_ptile'))
            .otherwise(pl.col('mean_pheno_value_ptile')),
    )
    
    .group_by(["annotation", "region", "gene_name", "phenotype"])
    .agg(
        n_dis_above_cutoff = (
            (pl.col("annotation_score_dircor") == 1) & 
            (pl.col("mean_pheno_value_dircor_ptile") >= or_threshold_pheno)
        ).sum(),
        n_notdis_above_cutoff = (
            (pl.col("annotation_score_dircor") == 1) & 
            (pl.col("mean_pheno_value_dircor_ptile") < or_threshold_pheno)
        ).sum(),
        n_dis_below_cutoff = (
            (pl.col("annotation_score_dircor") == 0) & 
            (pl.col("mean_pheno_value_dircor_ptile") >= or_threshold_pheno)
        ).sum(),
        n_notdis_below_cutoff = (
            (pl.col("annotation_score_dircor") == 0) & 
            (pl.col("mean_pheno_value_dircor_ptile") < or_threshold_pheno)
        ).sum()
    )
    .with_columns(
        # odds_ratio = ((pl.col("n_dis_above_cutoff") + 1) / (pl.col("n_notdis_above_cutoff")+1)) / (1 - or_threshold_pheno)
        odds_ratio = (pl.col("n_dis_above_cutoff") / pl.col("n_notdis_above_cutoff")) / (pl.col("n_dis_below_cutoff") / pl.col("n_notdis_below_cutoff"))
    )
)

# Execute
print("Executing with odds ratios...")
or_df = gp_lazy.collect(engine='streaming')
or_df

Executing with odds ratios...


annotation,region,gene_name,phenotype,n_dis_above_cutoff,n_notdis_above_cutoff,n_dis_below_cutoff,n_notdis_below_cutoff,odds_ratio
str,str,str,str,u64,u64,u64,u64,f64
"""utr5_indel""","""ENSG00000258366""","""RTEL1""","""red_blood_cell_erythrocyte_cou…",0,0,814,16035,NaN
"""noncoding_indel""","""ENSG00000070182""","""SPTB""","""reticulocyte_percentage_int""",120,2583,1647,29982,0.845715
"""utr5_indel""","""ENSG00000196712""","""NF1""","""shbg_int""",0,0,3309,60074,NaN
"""am_pathogenicity""","""ENSG00000116183""","""PAPPA2""","""standing_height_int""",9,39,4774,86016,4.157907
"""intronic_indel""","""ENSG00000196739""","""COL27A1""","""trunk_fatfree_mass_int""",157,2457,1909,34995,1.171371
…,…,…,…,…,…,…,…,…
"""intronic_indel""","""ENSG00000101745""","""ANKRD12""","""red_blood_cell_erythrocyte_dis…",237,4364,2126,36350,0.928549
"""utr5_indel""","""ENSG00000159899""","""NPR2""","""trunk_predicted_mass_int""",0,0,342,5730,NaN
"""utr5_indel""","""ENSG00000166819""","""PLIN1""","""hdl_cholesterol_int""",0,0,281,5703,NaN


In [67]:
(    
    or_df
    .filter(
        (pl.col('annotation') == 'loftee_hc') &
        (pl.col("odds_ratio").is_finite())
    )
    .drop_nans()
    
)

annotation,region,gene_name,phenotype,n_dis_above_cutoff,n_notdis_above_cutoff,n_dis_below_cutoff,n_notdis_below_cutoff,odds_ratio
str,str,str,str,u64,u64,u64,u64,f64
"""loftee_hc""","""ENSG00000155903""","""RASA2""","""lymphocyte_count_int""",23,121,1688,30956,3.485899
"""loftee_hc""","""ENSG00000107863""","""ARHGAP21""","""leg_fatfree_mass_right_int""",9,28,1839,34605,6.048415
"""loftee_hc""","""ENSG00000127585""","""FBXL16""","""whole_body_fat_mass_int""",4,4,389,6963,17.899743
"""loftee_hc""","""ENSG00000104870""","""FCGRT""","""calcium_int""",8,5,415,7825,30.168675
"""loftee_hc""","""ENSG00000244734""","""HBB""","""mean_sphered_cell_volume_int""",9,5,223,3965,32.004484
…,…,…,…,…,…,…,…,…
"""loftee_hc""","""ENSG00000162551""","""ALPL""","""alkaline_phosphatase_int""",37,6,1188,17994,93.403199
"""loftee_hc""","""ENSG00000146197""","""SCUBE3""","""leg_predicted_mass_left_int""",5,34,561,9904,2.596204
"""loftee_hc""","""ENSG00000080345""","""RIF1""","""forced_vital_capacity_fvc_int""",11,49,1416,26063,4.131976


In [68]:
plt_df = (
    or_df
    .drop_nans()
    .filter(pl.col("odds_ratio").is_finite())
    .with_columns(
        n_gene_phenos = pl.len().over('annotation'),
        med_odds_ratio = pl.col('odds_ratio').median().over('annotation'),
        avg_odds_ratio = pl.col('odds_ratio').mean().over('annotation'),
        std_odds_ratio = pl.col('odds_ratio').std().over('annotation'),
    )
    .with_columns(
        std_ci_upper = pl.col('avg_odds_ratio') + 1.96 * (pl.col('std_odds_ratio')),
        std_ci_lower = pl.col('avg_odds_ratio') - 1.96 * (pl.col('std_odds_ratio')),
    )
    .select(['annotation', 'n_gene_phenos', 'med_odds_ratio', 'avg_odds_ratio', 'std_odds_ratio', 'std_ci_upper', 'std_ci_lower'])
    .unique()
)
plt_df

annotation,n_gene_phenos,med_odds_ratio,avg_odds_ratio,std_odds_ratio,std_ci_upper,std_ci_lower
str,u64,f64,f64,f64,f64,f64
"""utr3_indel""",45,0.0,0.763942,1.133648,2.985892,-1.458008
"""utr5_indel""",14,0.0,0.417885,0.854846,2.093383,-1.257613
"""loftee_hc""",1173,3.728214,9.468731,24.415344,57.322804,-38.385342
"""am_pathogenicity""",1055,2.290545,4.130714,8.591089,20.969248,-12.70782
"""intronic_indel""",1144,0.945846,0.950659,0.326459,1.590518,0.3108
"""noncoding_indel""",1176,0.937843,0.935231,0.16711,1.262766,0.607695


In [69]:
melted_anno.select(['id', 'annotation', 'annotation_score']).unique().filter(pl.col('annotation') == 'noncoding_indel').select('annotation_score').sum()

annotation_score
f32
2.338927e6


In [32]:
melted_anno.select(['id', 'annotation', 'annotation_score']).unique().filter(pl.col('annotation') == 'am_pathogenicity').select('annotation_score').sum()

annotation_score
f32
27778.0


In [33]:
melted_anno.select(['id', 'annotation', 'annotation_score']).unique().filter(pl.col('annotation') == 'loftee_hc').select('annotation_score').sum()

annotation_score
f32
21519.0
